In [1]:
%pwd

'c:\\Users\\mnsnn\\Documents\\Learn Advanced\\ML - MLOps\\datascienceproject\\research'

In [2]:
import os
os.chdir('../')

%pwd

'c:\\Users\\mnsnn\\Documents\\Learn Advanced\\ML - MLOps\\datascienceproject'

In [3]:
# entity/config_entity.py
from dataclasses import dataclass
from pathlib import Path

@dataclass
class ModelTrainingConfig:
  root_dir: Path
  train_data: Path
  test_data: Path
  model_name: str
  alpha: float
  l1_ratio: float
  target: str

In [4]:
# configs/configurations.py

from src.datascienceproject.constants import PARAMS_FILE_PATH
from src.datascienceproject.constants import SCHEMA_FILE_PATH
from src.datascienceproject.constants import CONFIG_FILE_PATH

from src.datascienceproject.utils.common import read_yaml, create_directories

class ConfigurationManager:
    
	def __init__(self,
		params_file_path = PARAMS_FILE_PATH,
		config_file_path = CONFIG_FILE_PATH,
		schema_file_path = SCHEMA_FILE_PATH):

		# Loading yaml files
		self.config = read_yaml(config_file_path)
		self.params = read_yaml(params_file_path)
		self.schema = read_yaml(schema_file_path)
		
		# Creating artifact root dir
		create_directories([self.config.artifacts_root])

	def get_model_training_configs(self):

		config = self.config.model_training
		params = self.params.ElasticNet
		schema = self.schema.TARGET_COLUMN

		create_directories([config.root_dir])

		model_training_configs = ModelTrainingConfig(
			root_dir = config.root_dir,
			train_data = config.train_data,
			test_data = config.test_data,
			model_name = config.model_name,
			alpha = params.alpha,
			l1_ratio = params.l1_ratio,
			target = schema.name
		)

		return model_training_configs


In [ ]:
# components/model_training.py
import pandas as pd
import numpy as np
import os
from sklearn.linear_model import ElasticNet
import joblib
from src.datascienceproject import logging


class ModelTraining:
    
	def __init__(self, config:ModelTrainingConfig) -> ModelTrainingConfig:
		self.config = config

	def train(self):

		# Loading the data
		train_data = pd.read_csv(self.config.train_data)
		test_data = pd.read_csv(self.config.test_data)

		# Splitting to X and Y splits
		X_train = train_data.drop(columns=[self.config.target])
		y_train = train_data[self.config.target]
		X_test = test_data.drop(columns=[self.config.target])
		y_test = test_data[self.config.target]

		# Configuring the model
		model = ElasticNet(
			alpha=self.config.alpha,
			l1_ratio=self.config.l1_ratio,
			random_state=42
		)

		# Training the model
		model.fit(X_train, y_train)

		# Saving the model pickle
		joblib.dump(model, os.path.join(self.config.root_dir, self.config.model_name))

In [6]:
config = ConfigurationManager()
model_training_config = config.get_model_training_configs()
model_training = Modeltraining(config=model_training_config)
model_training.train()

2026-05-26 21:50:06,893 : INFO : YAML file config\config.yaml loaded successfully
2026-05-26 21:50:06,896 : INFO : YAML file params.yaml loaded successfully
2026-05-26 21:50:06,899 : INFO : YAML file schema.yaml loaded successfully
2026-05-26 21:50:06,900 : INFO : Created directory at artifacts
2026-05-26 21:50:06,902 : INFO : Created directory at artifacts/model_training
